In [1]:
library(tidyverse)
library(edgeR)
library(ggrastr)
library(gridExtra)
library(RColorBrewer)
library(patchwork)
theme_set(theme_bw())

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: limma


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




In [2]:
#read in counts data
countmat <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/brm10uM_timecourse/timecourse_countmat.tsv", header=F, sep='\t')
#read in row metadata
row_meta <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/bed/ATAC_peaks/filtered/consensus_filt_annot.tsv", sep='\t')

#read in column metadata
col_meta <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/brm10uM_timecourse/column_metadata.txt", sep='\t')

#filter to only include counts (exclude genomic coordinates)
counts <- countmat[,5:30]

In [3]:
#read in control counts (reads over promoters of genes invariantly expressed in BAFinh 6h/10µM vs DMSO
control_mat <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/brm10uM_timecourse/timecourse_control_countmat.tsv", header=F, sep='\t')
#calculate effective library sizes for normalization (sum of counts over each control region set)
lib_sizes <- colSums(control_mat[,5:30])

#add library sizes to column metadata
col_meta <- col_meta %>% mutate(sizes=lib_sizes)

In [4]:
# Differential Accessibility Region (DAR) Analysis with Custom Normalization
#
# This function performs differential accessibility analysis using edgeR,
# incorporating custom normalization based on user-supplied effective library sizes 
# The specific use case is for normalizing ATAC-seq using counts from promoter regions of genes that are invariantly expressed in control and BAF-inhibited samples.
#
# Inputs are:
#     data,  A matrix of raw counts (peaks x samples).
#     coldata, A data frame containing sample metadata. Must include:
#      `Condition`: Factor or character vector indicating experimental groups (e.g., control, treatment).
#      `sizes`: Numeric vector of effective library sizes (i.e., sum of counts at invariant promoters)
#
# This function returns a data frame with differential test results for each region, including logFC, p-values, and FDR
# As well as a classification of each element as gaining ("up"), losing ("down"), or not changing ("nc") in accessibility
# Based on FDR and logFC cutoffs of 0.05 and log2(1.5), respectively

dar <- function(data, coldata) {
    library(edgeR)
  
  # Create DGEList object with counts and group information
    dge <- DGEList(counts = data, group = coldata$Condition)
  
  # Override default library size with effective library sizes from control regions
    dge$samples$lib.size <- coldata$sizes
  
  # Set normalization factors to 1 (i.e., use lib.size as-is with no further scaling)
    dge$samples$norm.factors <- rep(1, nrow(coldata))
  
  # Create design matrix for comparison (model without intercept, group-specific columns)
    design <- model.matrix(~0 + group, data = dge$samples)
    colnames(design) <- levels(dge$samples$group)  # optional: label columns by condition names
  
  # Estimate dispersion 
    dge <- estimateDisp(dge, design)
  
  # Fit generalized linear model (GLM) 
    dge_fit <- glmQLFit(dge, design)
  
  # Conduct differential test (treatment vs control)
    diff <- glmQLFTest(dge_fit, contrast = c(1, -1))
  
  # Return all regions without sorting; include logFC, p-value, FDR, etc.
    res <- topTags(diff, n = Inf, sort.by = "none")$table 
  # Add column categorizing peaks as gaining, losing, or not changing accessibility
    res <- res %>% mutate(
                    change = case_when(FDR < 0.05 & logFC > log2(2) ~ "up",
                                       FDR < 0.05 & logFC < -log2(2) ~ "down",
                                       TRUE ~ "nc"))
  return(res)
}

# This function implements the dar() function separately for TSS-distal and TSS-proximal peaks. 
# In addition to the data and coldata params required by dar(), this function additionall requires
# a 'rowdata' object (should contain minimally genomic coordinates and TSS proximity for each row)

# This function returns a list object where the first element is the table of results from DAR analysis, 
# And the second is a reformatted rowdata object (rows reordered so that they match the results table)

dar_splitTSS <- function(data, coldata, rowdata){ 
    #Use row metadata to split counts matrix into TSS-proximal and dista
    #Run DAR analysis separately on each
    res_tss <- dar(data=data[rowdata$TSS==TRUE,], coldata)
    res_nontss <- dar(data=data[rowdata$TSS==FALSE,], coldata)

    #combine TSS and nonTSS DAR results
    res <- rbind(res_tss, res_nontss) %>% mutate(BAFdep=case_when(change == "down" ~ TRUE, TRUE ~ FALSE)) #add column that binarizes peaks into BAF-dependent (significant accessibility loss) and independent
    #reorder rowdata
    rowdata <- rbind(rowdata[rowdata$TSS==TRUE,], rowdata[rowdata$TSS==FALSE,])
    return(list("results"=res, "row_meta"=rowdata))
}

In [5]:
#Run DAR analysis for each BAFinh timepoint

dar_0.5h <- dar_splitTSS(data=counts[,col_meta$Time==0.5], coldata=col_meta[col_meta$Time==0.5,], rowdata=row_meta)
dar_1h <- dar_splitTSS(data=counts[,col_meta$Time==1], coldata=col_meta[col_meta$Time==1,], rowdata=row_meta)
dar_2h <- dar_splitTSS(data=counts[,col_meta$Time==2], coldata=col_meta[col_meta$Time==2,], rowdata=row_meta)
dar_6h <- dar_splitTSS(data=counts[,col_meta$Time==6], coldata=col_meta[col_meta$Time==6,], rowdata=row_meta)
dar_24h <- dar_splitTSS(data=counts[,col_meta$Time==24], coldata=col_meta[col_meta$Time==24,], rowdata=row_meta)

In [6]:
# write out DAR results as table
#attach row metadata to DAR results table when writing out
row_meta_slim <- dar_0.5h$row_meta %>% select(seqnames, start, end, TSS, type, ID)

write.table(cbind(row_meta_slim, dar_0.5h$results), "time_0.5h_DARs.tsv", sep='\t', row.names=FALSE, quote=FALSE)
write.table(cbind(row_meta_slim, dar_1h$results), "time_1h_DARs.tsv", sep='\t', row.names=FALSE, quote=FALSE)
write.table(cbind(row_meta_slim, dar_2h$results), "time_2h_DARs.tsv", sep='\t', row.names=FALSE, quote=FALSE)
write.table(cbind(row_meta_slim, dar_6h$results), "time_6h_DARs.tsv", sep='\t', row.names=FALSE, quote=FALSE)
write.table(cbind(row_meta_slim, dar_24h$results), "time_24h_DARs.tsv", sep='\t', row.names=FALSE, quote=FALSE)

In [15]:
dar_0.5h$results %>% group_by(change) %>% dplyr::summarize(n())
dar_1h$results %>% group_by(change) %>% dplyr::summarize(n())
dar_2h$results %>% group_by(change) %>% dplyr::summarize(n())
dar_6h$results %>% group_by(change) %>% dplyr::summarize(n())
dar_24h$results %>% group_by(change) %>% dplyr::summarize(n())

change,n()
<chr>,<int>
down,52207
nc,59488
up,26


change,n()
<chr>,<int>
down,50411
nc,61136
up,174


change,n()
<chr>,<int>
down,52012
nc,59471
up,238


change,n()
<chr>,<int>
down,50658
nc,60594
up,469


change,n()
<chr>,<int>
down,57434
nc,53917
up,370
